# 04 統計分析

Steamゲームのプレイヤー数の衰退について、ジャンル差と関連要因を探索的に検討する。

## 分析上の注意

- 分析単位はゲームであり、同じゲームの月次データを独立標本として水増ししない。
- 対象は12作品、各ジャンル2作品の便宜的標本であるため、検定力と一般化可能性は限定的である。
- `decline_rate` は観測期間中のピークから最新月までの減少率である。ピークを事後的に選ぶため、値には構造的な上方バイアスがあり得る。
- レビュー集計は取得時点までの累積値であり、衰退の前後関係や因果関係は検証できない。
- 有意水準は5%とするが、p値だけでなく効果量と標本数も併記する。3検定についてHolm法で多重比較補正したp値も示す。

In [ ]:
# 必要なライブラリを読み込む
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# 日本語フォントが環境にあれば優先して使用する
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "IPAexGothic", "Yu Gothic", "Hiragino Sans", "sans-serif"]
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

ALPHA = 0.05
DATA_DIR = Path("/content/drive/MyDrive/卒業研究/steam_research/data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

print(f"データ読込先: {DATA_DIR.resolve()}")

In [ ]:
# 数値列に含まれるカンマや%記号を除いて数値化する
def to_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip(),
        errors="coerce",
    )


# 月次プレイヤー数から、ゲームごとの衰退指標を作る
monthly = pd.read_csv(DATA_DIR / "steamcharts_monthly.csv", encoding="utf-8-sig")
category_col = "category" if "category" in monthly.columns else "genre"
monthly = monthly[monthly["month"] != "Last 30 Days"].copy()
monthly["month_date"] = pd.to_datetime(monthly["month"], format="%B %Y", errors="coerce")
for column in ["avg_players", "peak_players"]:
    monthly[column] = to_number(monthly[column])
monthly = monthly.dropna(subset=["appid", "month_date", "avg_players"])
monthly["appid"] = monthly["appid"].astype(int)

rows = []
for appid, game in monthly.groupby("appid"):
    game = game.sort_values("month_date").copy()
    peak_row = game.loc[game["avg_players"].idxmax()]
    latest_row = game.iloc[-1]
    game["previous_players"] = game["avg_players"].shift(1)
    game["monthly_drop_rate"] = (game["previous_players"] - game["avg_players"]) / game["previous_players"]
    post_peak = game.loc[game["month_date"] > peak_row["month_date"], "monthly_drop_rate"].replace([np.inf, -np.inf], np.nan).dropna()
    decline_rate = np.nan if peak_row["avg_players"] <= 0 else (peak_row["avg_players"] - latest_row["avg_players"]) / peak_row["avg_players"]
    rows.append({
        "appid": int(appid),
        "name": latest_row["name"],
        "genre": latest_row[category_col],
        "peak_avg_players": peak_row["avg_players"],
        "latest_avg_players": latest_row["avg_players"],
        "decline_rate": decline_rate,
        "largest_monthly_drop_rate": post_peak.max() if len(post_peak) else np.nan,
        "months_observed": len(game),
    })
game_df = pd.DataFrame(rows)

# Steamの累積レビュー集計から低評価率を作る
reviews = pd.read_csv(DATA_DIR / "review_summary.csv", encoding="utf-8-sig")
reviews["negative_rate"] = reviews["total_negative"] / (reviews["total_positive"] + reviews["total_negative"])
review_by_game = (
    reviews.sort_values("collected_at")
    .drop_duplicates("appid", keep="last")
    [["appid", "negative_rate", "total_reviews"]]
)
analysis_df = game_df.merge(review_by_game, on="appid", how="left")
display(analysis_df.sort_values(["genre", "name"]).reset_index(drop=True))
print(f"分析対象: {len(analysis_df)}作品 / レビュー結合成功: {analysis_df['negative_rate'].notna().sum()}作品")

In [ ]:
# 卒論本文・発表スライド向けの定型出力を行う関数
def print_test_summary(test_name, null_hypothesis, statistic_text, p_value, effect_text, can_say, cannot_say, adjusted_p=None):
    # 多重比較補正値がある場合は、補正後p値で結論を判定する
    decision_p = adjusted_p if adjusted_p is not None else p_value
    significant = decision_p < ALPHA
    result = "5%水準で統計的に有意であった" if significant else "5%水準で統計的に有意ではなかった"
    print("\n" + "=" * 72)
    print(f"何を検定したか: {test_name}")
    print(f"帰無仮説: {null_hypothesis}")
    print(f"結果: {result}（{statistic_text}）")
    print(f"p値: {p_value:.4f}")
    if adjusted_p is not None:
        print(f"Holm法による補正p値: {adjusted_p:.4f}")
    print(f"効果量: {effect_text}")
    print(f"何が言えるか: {can_say}")
    print(f"何は言えないか: {cannot_say}")


# Holm法で多重比較を補正する（外部ライブラリに依存しない実装）
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted_sorted = np.maximum.accumulate((len(p_values) - np.arange(len(p_values))) * p_values[order])
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted


# 3つの検定を先に計算し、共通の多重比較補正を適用する
genre_groups = [group["decline_rate"].dropna().values for _, group in analysis_df.groupby("genre")]
kw_h, kw_p = stats.kruskal(*genre_groups)
kw_n = sum(len(group) for group in genre_groups)
kw_k = len(genre_groups)
# epsilon squared。標本変動により負になった場合は0に丸める
kw_epsilon_sq = max(0.0, (kw_h - kw_k + 1) / (kw_n - kw_k)) if kw_n > kw_k else np.nan

peak_part = analysis_df[["peak_avg_players", "decline_rate"]].dropna()
peak_rho, peak_p = stats.spearmanr(peak_part["peak_avg_players"], peak_part["decline_rate"])
review_part = analysis_df[["negative_rate", "decline_rate"]].dropna()
review_rho, review_p = stats.spearmanr(review_part["negative_rate"], review_part["decline_rate"])
adjusted_p_values = holm_adjust([kw_p, peak_p, review_p])
print("3検定の計算とHolm法による多重比較補正が完了しました。")

In [ ]:
# 分析1: ジャンル別の衰退率を可視化する
genre_order = analysis_df.groupby("genre")["decline_rate"].median().sort_values().index
plot_data = [analysis_df.loc[analysis_df["genre"] == genre, "decline_rate"].dropna() for genre in genre_order]
plt.boxplot(plot_data, tick_labels=genre_order, showmeans=True)
plt.ylabel("ピークから最新月までの衰退率")
plt.xlabel("ジャンル")
plt.title("ジャンル別の衰退率")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print_test_summary(
    test_name="6ジャンル間でゲームの衰退率の分布が異なるか（Kruskal–Wallis検定）",
    null_hypothesis="すべてのジャンルで衰退率の分布は同じである。",
    statistic_text=f"H({kw_k - 1}) = {kw_h:.3f}, n = {kw_n}",
    p_value=kw_p,
    adjusted_p=adjusted_p_values[0],
    effect_text=f"epsilon squared = {kw_epsilon_sq:.3f}",
    can_say=("Holm補正後も、この標本ではジャンル間の衰退率に統計的な差が示された。" if adjusted_p_values[0] < ALPHA else "Holm補正後、この標本ではジャンル間の衰退率の差を統計的に確認できなかった。"),
    cannot_say="ジャンルごとに2作品しかないため、差がないとは断定できず、Steam上の各ジャンル全体にも一般化できない。どのジャンル間に差があるかも、この全体検定だけでは特定できない。",
)

In [ ]:
# 分析2: ピーク時プレイヤー規模と衰退率の単調な関連を調べる
plt.scatter(peak_part["peak_avg_players"], peak_part["decline_rate"], alpha=0.8)
for _, row in analysis_df.dropna(subset=["peak_avg_players", "decline_rate"]).iterrows():
    plt.annotate(row["name"], (row["peak_avg_players"], row["decline_rate"]), fontsize=7, alpha=0.8)
plt.xscale("log")
plt.xlabel("ピーク時の平均プレイヤー数（対数軸）")
plt.ylabel("ピークから最新月までの衰退率")
plt.title("ピーク規模と衰退率の関係")
plt.tight_layout()
plt.show()

direction = "正" if peak_rho > 0 else "負"
print_test_summary(
    test_name="ピーク時の平均プレイヤー数と衰退率に単調な関連があるか（Spearmanの順位相関検定）",
    null_hypothesis="ピーク時の平均プレイヤー数と衰退率の母順位相関係数は0である。",
    statistic_text=f"Spearmanのrho = {peak_rho:.3f}, n = {len(peak_part)}",
    p_value=peak_p,
    adjusted_p=adjusted_p_values[1],
    effect_text=f"Spearmanのrho = {peak_rho:.3f}（{direction}の関連）",
    can_say=(f"Holm補正後も、この標本ではピーク規模と衰退率の間に統計的に有意な{direction}の順位相関がみられた。" if adjusted_p_values[1] < ALPHA else f"この標本では{direction}の相関係数が得られたが、Holm補正後に統計的に有意な関連は確認できなかった。"),
    cannot_say="ピーク規模が衰退を引き起こすとは言えない。発売時期、運営期間、作品特性などの交絡を調整しておらず、標本も12作品に限られる。",
)

In [ ]:
# 分析3: 累積低評価率と衰退率の単調な関連を調べる
plt.scatter(review_part["negative_rate"], review_part["decline_rate"], alpha=0.8)
for _, row in analysis_df.dropna(subset=["negative_rate", "decline_rate"]).iterrows():
    plt.annotate(row["name"], (row["negative_rate"], row["decline_rate"]), fontsize=7, alpha=0.8)
plt.xlabel("Steam累積レビューの低評価率")
plt.ylabel("ピークから最新月までの衰退率")
plt.title("低評価率と衰退率の関係")
plt.tight_layout()
plt.show()

review_direction = "正" if review_rho > 0 else "負"
print_test_summary(
    test_name="Steam累積レビューの低評価率と衰退率に単調な関連があるか（Spearmanの順位相関検定）",
    null_hypothesis="累積低評価率と衰退率の母順位相関係数は0である。",
    statistic_text=f"Spearmanのrho = {review_rho:.3f}, n = {len(review_part)}",
    p_value=review_p,
    adjusted_p=adjusted_p_values[2],
    effect_text=f"Spearmanのrho = {review_rho:.3f}（{review_direction}の関連）",
    can_say=(f"Holm補正後も、この標本では累積低評価率と衰退率の間に統計的に有意な{review_direction}の順位相関がみられた。" if adjusted_p_values[2] < ALPHA else f"この標本では{review_direction}の相関係数が得られたが、Holm補正後に統計的に有意な関連は確認できなかった。"),
    cannot_say="低評価が衰退の原因だとは言えず、逆因果も否定できない。レビューは取得時点までの累積値で、衰退前後を分離していない。",
)

In [ ]:
# 今回の分析から卒論の「結果」「考察」に書ける内容を自動で整理する
print("\n【卒論の「結果」に書ける内容】")
print(f"- 12作品を対象に、ピークから最新月までの衰退率をゲーム単位で算出した。")
print(f"- ジャンル間の衰退率差: Kruskal–Wallis H({kw_k - 1}) = {kw_h:.3f}, p = {kw_p:.4f}, Holm補正p = {adjusted_p_values[0]:.4f}, epsilon squared = {kw_epsilon_sq:.3f}。")
print(f"- ピーク規模と衰退率: Spearmanのrho = {peak_rho:.3f}, p = {peak_p:.4f}, Holm補正p = {adjusted_p_values[1]:.4f}, n = {len(peak_part)}。")
print(f"- 累積低評価率と衰退率: Spearmanのrho = {review_rho:.3f}, p = {review_p:.4f}, Holm補正p = {adjusted_p_values[2]:.4f}, n = {len(review_part)}。")
print("- p値だけでなく効果量を併記し、3検定に対してHolm法による多重比較補正を行った。")

print("\n【卒論の「考察」に書ける内容】")
if adjusted_p_values[0] < ALPHA:
    print("- ジャンルによって衰退率の分布が異なる可能性が示唆された。ただし各ジャンル2作品であり、追試が必要である。")
else:
    print("- ジャンル差は統計的に確認できなかったが、小標本による検定力不足が考えられるため、『差がない』とは結論づけられない。")
if adjusted_p_values[1] < ALPHA:
    print(f"- ピーク規模と衰退率には{direction}の関連が示唆されたが、交絡要因を調整していないため因果的説明はできない。")
else:
    print("- ピーク規模だけでは衰退率を十分に説明できない可能性があるが、対象作品の追加と交絡要因の検討が必要である。")
if adjusted_p_values[2] < ALPHA:
    print(f"- 累積低評価率と衰退率には{review_direction}の関連が示唆されたが、時間的前後関係は特定できない。")
else:
    print("- 累積低評価率と衰退率の関連は統計的に確認できなかった。レビューの時期を衰退前後に分けた追加分析が望まれる。")
print("- 本分析は探索的分析であり、便宜的に選んだ12作品からSteamゲーム全体へ一般化することはできない。")
print("- 今後は対象作品数を増やし、発売後経過月数をそろえ、アップデート・価格施策・競合作品などを含む縦断分析を行う必要がある。")